# 3. 原始极性数据分箱（T=200）

按 `bin(t) = floor(t * T / 1000)` 分箱并对每个区间取最大值。

In [ ]:
# 修改 T 即可生成不同时间步长的分箱数据。
T = 200

# 仅当明确需要重新生成同一参数目录时改为 True。
OVERWRITE = False

In [ ]:
import hashlib
import json
import os
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import torch
from tqdm.auto import tqdm


def find_project_root():
    # 同时兼容从项目根目录或 notebook 所在目录启动 Jupyter。
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src' / 'data_prepare').is_dir() and (candidate / 'CapgMyo_data').is_dir():
            return candidate
    raise RuntimeError('找不到同时包含 src/data_prepare 和 CapgMyo_data 的项目根目录。')


if isinstance(T, bool) or not isinstance(T, int) or not 2 <= T <= 999:
    raise ValueError(f'T 必须是 [2, 999] 范围内的整数，当前为 {T!r}。')

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / 'CapgMyo_data'
SOURCE_ROOT = DATA_ROOT / 'raw_polarity'
SOURCE_MANIFEST_PATH = SOURCE_ROOT / 'manifest.json'
OUTPUT_ROOT = DATA_ROOT / 'raw_polarity_binned' / f'T_{T}'
OUTPUT_MANIFEST_PATH = OUTPUT_ROOT / 'manifest.json'
SPLIT_NAMES = ('train', 'val', 'test')
EXPECTED_COUNTS = {'train': 1008, 'val': 144, 'test': 288}
SPLIT_REPETITIONS = {
    'train': {1, 3, 4, 5, 6, 7, 8},
    'val': {10},
    'test': {2, 9},
}
INPUT_CHANNELS = 2
METADATA_KEYS = ('label', 'subject_id', 'gesture_id', 'repetition_id')
FILE_NAME_PATTERN = re.compile(r's(\d+)_g(\d+)_r(\d+)\.pt$', re.IGNORECASE)
NATIVE_TIME_STEPS = 1000
TIME_BIN_INDEX = torch.div(
    torch.arange(NATIVE_TIME_STEPS, dtype=torch.long) * T,
    NATIVE_TIME_STEPS,
    rounding_mode='floor',
)

print(f'项目根目录：{PROJECT_ROOT}')
print(f'输出目录：{OUTPUT_ROOT}')
print(f'T={T}, OVERWRITE={OVERWRITE}')

In [ ]:
def load_pt(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        # 兼容尚未提供 weights_only 参数的旧版 PyTorch。
        return torch.load(path, map_location='cpu')


def write_json_atomic(path, value):
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    temporary_path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    os.replace(temporary_path, path)


def save_pt_atomic(path, payload):
    # 只在完整写入后替换目标，避免中断留下可见但不完整的样本。
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary_path)
    os.replace(temporary_path, path)


def sha256sum(path):
    digest = hashlib.sha256()
    with path.open('rb') as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def scalar_int(value, key, path):
    if not isinstance(value, torch.Tensor) or value.ndim != 0 or value.dtype != torch.int64:
        raise TypeError(f'{path} 中 {key} 必须是 torch.int64 标量。')
    return int(value)


def validate_metadata(payload, path):
    missing = {'data', *METADATA_KEYS} - set(payload)
    if missing:
        raise KeyError(f'{path} 缺少字段：{sorted(missing)}')

    metadata = {key: scalar_int(payload[key], key, path) for key in METADATA_KEYS}
    if metadata['label'] != metadata['gesture_id'] - 1 or not 0 <= metadata['label'] <= 7:
        raise ValueError(f'{path} 中 label/gesture_id 不合法。')
    if not 1 <= metadata['subject_id'] <= 18:
        raise ValueError(f'{path} 中 subject_id 越界。')
    if not 1 <= metadata['gesture_id'] <= 8:
        raise ValueError(f'{path} 中 gesture_id 越界。')
    if not 1 <= metadata['repetition_id'] <= 10:
        raise ValueError(f'{path} 中 repetition_id 越界。')

    match = FILE_NAME_PATTERN.fullmatch(path.name)
    if match is None:
        raise ValueError(f'文件名不符合 sXX_gXX_rXX.pt：{path.name}')
    file_metadata = tuple(map(int, match.groups()))
    payload_metadata = (
        metadata['subject_id'],
        metadata['gesture_id'],
        metadata['repetition_id'],
    )
    if file_metadata != payload_metadata:
        raise ValueError(f'{path} 的文件名与内部元数据不一致。')
    return metadata


def validate_source_payload(payload, path):
    if not isinstance(payload, dict):
        raise TypeError(f'{path} 的内容必须是字典。')
    metadata = validate_metadata(payload, path)
    data = payload['data']
    expected_shape = (NATIVE_TIME_STEPS, INPUT_CHANNELS, 8, 16)
    if not isinstance(data, torch.Tensor):
        raise TypeError(f'{path} 中 data 必须是 Tensor。')
    if tuple(data.shape) != expected_shape or data.dtype != torch.float32:
        raise ValueError(
            f'{path} 中 data 为 shape={tuple(data.shape)}, dtype={data.dtype}；'
            f'预期 shape={expected_shape}, dtype=torch.float32。'
        )
    if not torch.isfinite(data).all():
        raise ValueError(f'{path} 中 data 包含 NaN 或 Inf。')
    if torch.any(data < 0):
        raise ValueError(f'{path} 中 raw_polarity data 包含负值。')
    return data, metadata


def bin_max(data):
    # 每个原生时刻恰好属于一个区间；非整除 T 也不会重叠或遗漏。
    flat_data = data.flatten(start_dim=1)
    binned = torch.zeros((T, flat_data.shape[1]), dtype=flat_data.dtype)
    expanded_index = TIME_BIN_INDEX[:, None].expand_as(flat_data)
    if hasattr(binned, 'scatter_reduce_'):
        binned.scatter_reduce_(0, expanded_index, flat_data, reduce='amax', include_self=True)
    else:
        # 旧版 PyTorch 的兼容路径只循环 T 次，不改变区间定义。
        for bin_index in range(T):
            binned[bin_index] = flat_data[TIME_BIN_INDEX == bin_index].amax(dim=0)
    return binned.reshape(T, *data.shape[1:]).contiguous()


def metadata_values_equal(left, right):
    if isinstance(left, torch.Tensor) and isinstance(right, torch.Tensor):
        return torch.equal(left, right)
    return type(left) is type(right) and left == right


def validate_preserved_fields(source_payload, output_payload, path):
    source_keys = set(source_payload) - {'data'}
    output_keys = set(output_payload) - {'data'}
    if output_keys != source_keys:
        raise ValueError(f'{path} 未完整保留源样本的非 data 字段。')
    for key in source_keys:
        if not metadata_values_equal(source_payload[key], output_payload[key]):
            raise ValueError(f'{path} 中字段 {key} 与源样本不一致。')


def build_output_payload(source_payload, binned_data):
    preserved = {}
    for key, value in source_payload.items():
        if key == 'data':
            continue
        preserved[key] = value.clone() if isinstance(value, torch.Tensor) else value
    return {'data': binned_data, **preserved}


def validate_output_payload(source_payload, output_payload, path):
    if not isinstance(output_payload, dict):
        raise TypeError(f'{path} 的内容必须是字典。')
    validate_metadata(output_payload, path)
    validate_preserved_fields(source_payload, output_payload, path)
    data = output_payload['data']
    expected_shape = (T, INPUT_CHANNELS, 8, 16)
    if not isinstance(data, torch.Tensor):
        raise TypeError(f'{path} 中 data 必须是 Tensor。')
    if tuple(data.shape) != expected_shape or data.dtype != torch.float32:
        raise ValueError(
            f'{path} 中 data 为 shape={tuple(data.shape)}, dtype={data.dtype}；'
            f'预期 shape={expected_shape}, dtype=torch.float32。'
        )
    if not torch.isfinite(data).all():
        raise ValueError(f'{path} 中 data 包含 NaN 或 Inf。')
    return data


def output_is_valid(path, source_payload):
    if not path.is_file():
        return False
    try:
        validate_output_payload(source_payload, load_pt(path), path)
        return True
    except (EOFError, KeyError, OSError, RuntimeError, TypeError, ValueError):
        return False

In [ ]:
if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError(f'缺少数据目录：{SOURCE_ROOT}')
if not SOURCE_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f'缺少源数据清单：{SOURCE_MANIFEST_PATH}')

source_fingerprint = {
    'root': SOURCE_ROOT.relative_to(PROJECT_ROOT).as_posix(),
    'manifest': SOURCE_MANIFEST_PATH.relative_to(PROJECT_ROOT).as_posix(),
    'manifest_sha256': sha256sum(SOURCE_MANIFEST_PATH),
}
source_files = {}
for split_name in SPLIT_NAMES:
    files = sorted((SOURCE_ROOT / split_name).glob('subject_*/*.pt'))
    if len(files) != EXPECTED_COUNTS[split_name]:
        raise RuntimeError(
            f'{split_name} 共有 {len(files)} 个样本，预期 {EXPECTED_COUNTS[split_name]} 个。'
        )
    source_files[split_name] = files

print('源数据文件数量检查通过：')
print({name: len(paths) for name, paths in source_files.items()})

In [ ]:
def manifest_matches_existing(manifest):
    parameters = manifest.get('parameters', {})
    if parameters.get('time_steps') != T:
        return False
    if manifest.get('sources') != {'raw_polarity': source_fingerprint}:
        return False
    return True


existing_files = list(OUTPUT_ROOT.glob('**/*.pt')) if OUTPUT_ROOT.is_dir() else []
existing_manifest = None
if OUTPUT_MANIFEST_PATH.is_file():
    existing_manifest = json.loads(OUTPUT_MANIFEST_PATH.read_text(encoding='utf-8'))
    if not manifest_matches_existing(existing_manifest) and not OVERWRITE:
        raise RuntimeError(
            '输出目录已有参数或源指纹不匹配的 manifest；请更换参数目录，'
            '或确认后将 OVERWRITE 设为 True。'
        )
elif existing_files and not OVERWRITE:
    # 分箱是确定性操作：允许复用无 manifest 的既有输出，并在复核后补写 manifest。
    print('输出目录已有 .pt 文件但没有 manifest；将逐个校验并复用有效文件。')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
records = []
conversion_stats = Counter()
train_sum = 0.0
train_sum_sq = 0.0
train_value_count = 0
total_trials = sum(EXPECTED_COUNTS.values())

with tqdm(total=total_trials, unit='trial', desc='Raw Polarity Binning') as progress:
    for split_name in SPLIT_NAMES:
        output_split_root = OUTPUT_ROOT / split_name
        for source_path in source_files[split_name]:
            source_payload = load_pt(source_path)
            data, metadata = validate_source_payload(source_payload, source_path)
            if metadata['repetition_id'] not in SPLIT_REPETITIONS[split_name]:
                raise ValueError(
                    f"{source_path} 的 repetition_id={metadata['repetition_id']} "
                    f'不属于 {split_name}。'
                )

            relative_path = source_path.relative_to(SOURCE_ROOT / split_name)
            output_path = output_split_root / relative_path
            output_path.parent.mkdir(parents=True, exist_ok=True)

            if not OVERWRITE and output_is_valid(output_path, source_payload):
                output_payload = load_pt(output_path)
                binned_data = output_payload['data']
                conversion_stats['reused'] += 1
            else:
                binned_data = bin_max(data)
                output_payload = build_output_payload(source_payload, binned_data)
                validate_output_payload(source_payload, output_payload, output_path)
                save_pt_atomic(output_path, output_payload)
                conversion_stats['written'] += 1

            if split_name == 'train':
                values = binned_data.to(dtype=torch.float64)
                train_sum += values.sum().item()
                train_sum_sq += (values * values).sum().item()
                train_value_count += binned_data.numel()

            records.append({
                'source_path': source_path.relative_to(PROJECT_ROOT).as_posix(),
                'output_path': output_path.relative_to(PROJECT_ROOT).as_posix(),
                'split': split_name,
                **metadata,
                'data_shape': [T, INPUT_CHANNELS, 8, 16],
                'data_dtype': 'torch.float32',
            })
            progress.update(1)
            progress.set_postfix(split=split_name)

train_mean = train_sum / train_value_count
train_variance = train_sum_sq / train_value_count - train_mean * train_mean
train_std = float(max(train_variance, 1e-12) ** 0.5)

print(f'分箱完成：{dict(conversion_stats)}')
print(f'train 集分箱后 mean={train_mean:.6g}, std={train_std:.6g}（仅作参考，不做归一化）')

In [ ]:
observed_counts = {}
verification_counts = Counter()

with tqdm(total=total_trials, unit='trial', desc='全量复核') as progress:
    for split_name in SPLIT_NAMES:
        output_paths = sorted((OUTPUT_ROOT / split_name).glob('subject_*/*.pt'))
        observed_counts[split_name] = len(output_paths)
        if len(output_paths) != EXPECTED_COUNTS[split_name]:
            raise RuntimeError(
                f'{split_name} 输出 {len(output_paths)} 个样本，预期 {EXPECTED_COUNTS[split_name]} 个。'
            )

        for source_path in source_files[split_name]:
            relative_path = source_path.relative_to(SOURCE_ROOT / split_name)
            output_path = OUTPUT_ROOT / split_name / relative_path
            source_payload = load_pt(source_path)
            validate_source_payload(source_payload, source_path)
            output_payload = load_pt(output_path)
            validate_output_payload(source_payload, output_payload, output_path)
            verification_counts['validated'] += 1
            progress.update(1)

manifest = {
    'format_version': 1,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'dataset': 'CapgMyo DB-a raw polarity binned data',
    'output_root': OUTPUT_ROOT.relative_to(PROJECT_ROOT).as_posix(),
    'parameters': {
        'time_steps': T,
        'overwrite': OVERWRITE,
    },
    'sources': {'raw_polarity': source_fingerprint},
    'binning': {
        'temporal_binning': 'bin(t) = floor(t * T / 1000), non-overlapping maximum',
        'input_shape': [NATIVE_TIME_STEPS, INPUT_CHANNELS, 8, 16],
        'output_shape': [T, INPUT_CHANNELS, 8, 16],
        'output_dtype': 'torch.float32',
        'normalization': None,
        'deterministic': True,
    },
    'statistics': {
        'train_mean_for_reference': train_mean,
        'train_std_for_reference': train_std,
    },
    'split_counts': observed_counts,
    'verification': {
        'validated_trials': verification_counts['validated'],
        'expected_trials': total_trials,
        'metadata_preserved': True,
        'float32_dtype_validated': True,
    },
    'records': records,
}
write_json_atomic(OUTPUT_MANIFEST_PATH, manifest)

print('全量复核通过。')
print(f'输出文件数量：{observed_counts}')
print(f'manifest：{OUTPUT_MANIFEST_PATH}')